# Piloto — Focal Loss vs. Cross Entropy (BETO)

**Pregunta:** ¿reemplazar la función de error actual por **Focal Loss** (Lin et al., 2017) mejora a BETO en IroSvA?

Este notebook es un **piloto**: usa **solo el train oficial** y **no toca el test**. Reutiliza sin cambios la carga de datos, los grupos (`GROUP_ID`), el tokenizador, el modelo y el Trainer de `04a_BETO.ipynb`, y la **configuración final** de cada variante (`beto_config_final.json`). Lo único que cambia es la función de error.

**Función de error actual.** Entropía cruzada ponderada por clase (`balanced`) con *label smoothing* 0.1. Con dos clases, softmax + entropía cruzada es exactamente la **entropía cruzada binaria (BCE)**: la probabilidad de *irónico* es la sigmoide de la diferencia de los dos logits, así que la opción `CE` también representa a BCE.

**Focal Loss.** Multiplica la entropía cruzada de cada tweet por $(1-p_t)^\gamma$, donde $p_t$ es la probabilidad que el modelo da a la clase correcta:

$$FL = -\,\alpha_t\,(1-p_t)^{\gamma}\,\log p_t$$

Los tweets que el modelo ya clasifica con seguridad ($p_t \to 1$) casi no aportan, y el entrenamiento se concentra en los difíciles. Con $\gamma = 0$ es la entropía cruzada. $\alpha_t$ son los mismos pesos de clase `balanced` que ya se usan.

| Opción | Función de error | Pesos de clase | Label smoothing |
|---|---|---|---|
| `CE` | Entropía cruzada (actual; equivale a BCE) | balanced | 0.1 |
| `FL_g1` | Focal Loss, γ = 1 | balanced (α) | 0 |
| `FL_g2` | Focal Loss, γ = 2 (valor de Lin et al., 2017) | balanced (α) | 0 |
| `FL_g2_LS` | Focal Loss, γ = 2 | balanced (α) | 0.1 |

Focal Loss también reduce la sobreconfianza del modelo (Mukhoti et al., 2020), igual que el *label smoothing*; por eso `FL_g1` y `FL_g2` van sin *label smoothing* y `FL_g2_LS` comprueba si conviene combinarlos.

**Protocolo (igual para las 4 opciones).**

- Mismos **3 folds agrupados** del train que la optimización final (`StratifiedGroupKFold`, `random_state=SEED`) y misma semilla por fold → comparación **pareada**.
- Mismos hiperparámetros finales de cada variante; se entrena hasta `num_train_epochs` evaluando cada época, y la comparación principal usa la **época del refit** (`refit_epochs`), que es la que usa el modelo final.
- Con los 3 folds, cada tweet del train recibe una predicción de un modelo que no lo vio (*out-of-fold*, OOF). Se compara el **F1-Macro OOF** de cada opción con `CE` y se calcula un **intervalo bootstrap pareado del 95 %** (remuestreo de tweets).
- **Regla de decisión (fijada antes de ejecutar):** una opción de Focal Loss se adopta solo si su IC 95 % frente a `CE` queda **por encima de 0 en al menos 2 de las 3 variantes**.

Costo: 4 opciones × 3 variantes × 3 folds = **36 fine-tunings**. Si se interrumpe, al volver a ejecutar se reutilizan los folds ya terminados.

**Referencias.** Lin, T.-Y., Goyal, P., Girshick, R., He, K., y Dollár, P. (2017). Focal loss for dense object detection. *ICCV*, 2980–2988. — Mukhoti, J., Kulharia, V., Sanyal, A., Golodetz, S., Torr, P. H. S., y Dokania, P. K. (2020). Calibrating deep neural networks using focal loss. *NeurIPS 33*, 15288–15299.

## 1. Librerías y entorno

In [1]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Versiones: torch 2.14.0+cu130 | transformers 5.17.0 | optuna 5.0.0 | numpy 1.26.4 | pandas 3.0.3
CUDA: True
GPU: NVIDIA RTX 6000 Ada Generation
VRAM (GB): 48.0


## 2. Configuración del modelo (igual que `04a_BETO.ipynb`) y del piloto

In [2]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

DATA_DIR = Path("../../data/transformers")
RESULTS_DIR = Path("../../data/resultados_beto_grouped_5x3_tpe20")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"  # primero "prueba"; luego "completo" y reiniciar kernel

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
RUN_TAG = "b5x3t20"
RUN_DIR = PROGRESS_DIR / RUN_TAG / MODO
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = True
EJECUTAR_OPTIMIZACION_FINAL = True  # solo train; se ejecuta tras la Nested CV
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]

# Exportación del modelo final e interpretabilidad (SHAP)
SEMILLA_EXPORTACION = 42  # el modelo de esta semilla del test se guarda en modelo_final/
N_SHAP_POR_TIPO = 50  # tuits explicados por tipo (VP, VN, FP, FN) y variante
SHAP_BATCH = 32  # lote de inferencia de SHAP (bajar si falta memoria de GPU)
MIN_TEXTOS_TOKEN = 5  # un token entra al análisis global si aparece en ≥ 5 tuits explicados
TOP_TOKENS = 10  # tokens por sentido (irónico / no irónico) en la figura global
MAX_TOKENS_EJEMPLO = 40  # tokens mostrados en cada ejemplo local
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "BETO"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito: se buscan weight_decay y classifier_dropout;
# el resto queda fijo (justificación en la celda «Decisiones de entrenamiento»).
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [2e-5],
    "batch_size": [16],
    "num_train_epochs": [6],  # máximo; la época efectiva la decide el early stopping
    "weight_decay": [0.01, 0.1],
    "loss_mode": ["balanced"],  # pérdida ponderada por clase (la ironía es minoritaria)
    "classifier_dropout": [0.1, 0.2, 0.3],
    "label_smoothing": [0.1],  # solo en la pérdida de entrenamiento
    "lr_decay": [0.9],  # LR por capas: cada capa inferior × 0.9
    "dropout_interno": [0.2],  # dropout oculto y de atención del encoder
    "congelar_frac": [0.0],  # se ajustan todas las capas
    "mascara_tokens": [0.15],  # fracción de tokens reemplazados por [MASK] solo al entrenar
}

# Warmup explícito. En transformers 5.x se declara con `warmup_steps` (fracción en [0, 1)).
WARMUP_FRACCION = 0.0

# Early stopping de los Inner Folds: épocas sin mejora de F1 antes de detener.
PACIENCIA_EARLY_STOPPING = 2

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
    "label_smoothing",
    "lr_decay",
    "dropout_interno",
    "congelar_frac",
    "mascara_tokens",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][BETO] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][BETO] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(f"[CONFIG][BETO] fine-tunings teóricos={fits_teoricos} | nested_activo={EJECUTAR_NESTED}")
print(f"[CONFIG][BETO] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][BETO] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l2_b16_e6_w10_b_d3_s1_r9_i2_c25_m15" / "inner3" / "checkpoint-99999"
).resolve()

print(f"[CONFIG][BETO] longitud ruta checkpoint ejemplo={len(str(_ruta_ckpt_ejemplo))} caracteres")

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

[CONFIG][BETO] modo=completo | variantes=['mx', 'es', 'cu']
[CONFIG][BETO] Nested=5×3 | trials=20 | startup=5
[CONFIG][BETO] fine-tunings teóricos=915 | nested_activo=True
[CONFIG][BETO] RUN_DIR=C:\Users\I21309\Documents\DeteccionDeSarcasmo\data\resultados_beto_grouped_5x3_tpe20\seguimiento\b5x3t20\completo
[CONFIG][BETO] SEARCH_SPACE OK: ['batch_size', 'classifier_dropout', 'congelar_frac', 'dropout_interno', 'label_smoothing', 'learning_rate', 'loss_mode', 'lr_decay', 'mascara_tokens', 'num_train_epochs', 'preprocessing', 'weight_decay']
[CONFIG][BETO] longitud ruta checkpoint ejemplo=190 caracteres


In [3]:
# ============================================================
# CONFIGURACIÓN DEL PILOTO (solo train oficial; no toca el test)
# ============================================================
PILOTO_DIR = Path("../../data/pilotos_focal_loss") / NOMBRE_EXPERIMENTO.lower()
PILOTO_DIR.mkdir(parents=True, exist_ok=True)

# Los archivos auxiliares del entrenamiento (tmp, logs, auditoría de grupos) van a la carpeta
# del piloto, no a la carpeta de resultados finales del modelo.
RUN_DIR = PILOTO_DIR
TMP_DIR = PILOTO_DIR / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

RUTA_CONFIG_FINAL = RESULTS_DIR / f"{NOMBRE_EXPERIMENTO.lower()}_config_final.json"
if not RUTA_CONFIG_FINAL.exists():
    raise FileNotFoundError(f"Falta {RUTA_CONFIG_FINAL}: ejecuta antes el notebook 04 de {NOMBRE_EXPERIMENTO}.")
CONFIG_FINAL = json.loads(RUTA_CONFIG_FINAL.read_text(encoding="utf-8"))

OPCIONES = {
    "CE": {"perdida": "ce", "gamma": 0.0, "label_smoothing": 0.1,
           "nombre": "Cross Entropy ponderada + LS 0.1 (actual)"},
    "FL_g1": {"perdida": "focal", "gamma": 1.0, "label_smoothing": 0.0,
              "nombre": "Focal Loss γ=1 ponderada"},
    "FL_g2": {"perdida": "focal", "gamma": 2.0, "label_smoothing": 0.0,
              "nombre": "Focal Loss γ=2 ponderada"},
    "FL_g2_LS": {"perdida": "focal", "gamma": 2.0, "label_smoothing": 0.1,
                 "nombre": "Focal Loss γ=2 ponderada + LS 0.1"},
}
BASE = "CE"
VARIANTES_PILOTO = ["mx", "es", "cu"]
N_FOLDS = 3
N_BOOTSTRAP = 2000

print(f"[PILOTO][{NOMBRE_EXPERIMENTO}] salida = {PILOTO_DIR.resolve()}")
print(f"[PILOTO][{NOMBRE_EXPERIMENTO}] {len(OPCIONES)} opciones × {len(VARIANTES_PILOTO)} variantes × "
      f"{N_FOLDS} folds = {len(OPCIONES) * len(VARIANTES_PILOTO) * N_FOLDS} fine-tunings")
for v in VARIANTES_PILOTO:
    c = CONFIG_FINAL[v]
    print(f"  {v.upper()}: wd={c['weight_decay']} | classifier_dropout={c['classifier_dropout']} | "
          f"máscara={c.get('mascara_tokens', 0.0)} | épocas={c['num_train_epochs']} | refit={c['refit_epochs']}")

[PILOTO][BETO] salida = C:\Users\I21309\Documents\DeteccionDeSarcasmo\data\pilotos_focal_loss\beto
[PILOTO][BETO] 4 opciones × 3 variantes × 3 folds = 36 fine-tunings
  MX: wd=0.01 | classifier_dropout=0.1 | máscara=0.15 | épocas=6 | refit=5
  ES: wd=0.1 | classifier_dropout=0.1 | máscara=0.15 | épocas=6 | refit=6
  CU: wd=0.1 | classifier_dropout=0.3 | máscara=0.15 | épocas=6 | refit=6


## 3. Datos, tokenizador y grupos (igual que el notebook 04)

In [4]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][BETO][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

[DATA][BETO][MX] n=2399 | clases={0: 1599, 1: 800} | manifest=OK
[DATA][BETO][ES] n=2397 | clases={0: 1598, 1: 799} | manifest=OK
[DATA][BETO][CU] n=2400 | clases={0: 1600, 1: 800} | manifest=OK


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

Tokenizer: BertTokenizer
do_lower_case: False


In [6]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_beto(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_beto(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][BETO][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

,variante,representacion,filas_en_repeticiones,grupos_repetidos,n_train,n_group_id,max_group_size,grupos_con_labels_mixtos
0,mx,ORIGINAL,0,0,2399,2399,1,0
1,mx,PYSENTIMIENTO,0,0,2399,2399,1,0
2,mx,TOK_PYSENTIMIENTO,0,0,2399,2399,1,0
3,es,ORIGINAL,0,0,2397,2307,17,1
4,es,PYSENTIMIENTO,131,41,2397,2307,17,1
5,es,TOK_PYSENTIMIENTO,131,41,2397,2307,17,1
6,cu,ORIGINAL,6,3,2400,2396,2,0
7,cu,PYSENTIMIENTO,6,3,2400,2396,2,0
8,cu,TOK_PYSENTIMIENTO,8,4,2400,2396,2,0


[AUDIT][BETO][MX] n=2399 | groups=2399 | max_group=1 | labels_mixtos=0
[AUDIT][BETO][ES] n=2397 | groups=2307 | max_group=17 | labels_mixtos=1
  ADVERTENCIA: 1 GROUP_ID contienen etiquetas mixtas. No es data leakage porque el grupo permanece íntegro dentro de un fold, pero representa ambigüedad/ruido de etiqueta y debe documentarse.
[AUDIT][BETO][CU] n=2400 | groups=2396 | max_group=2 | labels_mixtos=0


### 3.1 Modelo, Trainer y argumentos de entrenamiento (igual que el notebook 04)

In [7]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def softmax_np(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def nll_binaria(probs, labels):
    """NLL estándar: SIN pesos de clase y SIN label smoothing."""
    labels = np.asarray(labels, dtype=int)
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))


def ece_binaria(probs, labels, n_bins=10):
    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""
    labels = np.asarray(labels, dtype=int)
    conf = probs.max(axis=1)
    acierto = (probs.argmax(axis=1) == labels).astype(float)
    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)
    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())
                     for g in grupos if len(g)))


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    probs = softmax_np(logits)
    return {
        "nll": nll_binaria(probs, labels),
        "ece": ece_binaria(probs, labels),
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


PATRON_CAPA = re.compile(r"\.layer\.(\d+)\.")


def congelar_fraccion_inferior(model, fraccion):
    """congela embeddings + la fracción inferior de capas del encoder."""
    if not fraccion:
        return []
    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()
                      if (m := PATRON_CAPA.search(n)))
    n_congeladas = max(1, round(n_total * float(fraccion)))
    capas = set()
    for nombre, p in model.named_parameters():
        m = PATRON_CAPA.search(nombre)
        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):
            p.requires_grad = False
            if m:
                capas.add(int(m.group(1)))
    return sorted(capas)


def crear_modelo(seed, classifier_dropout=0.2, cfg=None):
    """`cfg` aporta `dropout_interno` y `congelar_frac`."""
    cfg = cfg or {}
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "bert":
            raise ValueError(f"Se esperaba arquitectura BERT, recibida: {config.model_type}")
        # BERT: dropout de la cabeza de clasificación.
        config.classifier_dropout = float(classifier_dropout)
        # dropout interno (oculto y de atención) del encoder y los embeddings.
        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "hidden_dropout_prob")))
        config.hidden_dropout_prob = dropout_interno
        config.attention_probs_dropout_prob = dropout_interno
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))
    return model


class ClasificadorTrainer(Trainer):
    """pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas
    + enmascarado aleatorio de tokens."""

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,
                 mascara_tokens=0.0, **kwargs):
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)
        self.lr_decay = float(lr_decay)
        self.mascara_tokens = float(mascara_tokens)
        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))
        super().__init__(*args, **kwargs)

    def _enmascarar(self, inputs):
        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el
        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""
        ids = inputs["input_ids"].clone()
        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))
        if "attention_mask" in inputs:
            candidatos &= inputs["attention_mask"].bool()
        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens
        ids[candidatos & sorteo] = tokenizer.mask_token_id
        return {**inputs, "input_ids": ids}

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if model.training and self.mascara_tokens > 0:
            inputs = self._enmascarar(inputs)
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        weights = (
            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        # Smoothing SOLO al entrenar: la pérdida de validación sigue siendo entropía cruzada estándar.
        ls = self.label_smoothing if model.training else 0.0
        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self, *args, **kwargs):
        if self.optimizer is not None:
            return self.optimizer
        if self.lr_decay >= 1.0:
            return super().create_optimizer(*args, **kwargs)
        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.
        patron = re.compile(r"\.layer\.(\d+)\.")
        n_capas = 1 + max(
            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))
        )
        base_lr, wd = self.args.learning_rate, self.args.weight_decay
        grupos = {}
        for nombre, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            m = patron.search(nombre)
            if m:
                profundidad = n_capas - int(m.group(1))  # última capa = 1
            elif "embeddings" in nombre:
                profundidad = n_capas + 1  # embeddings = LR más bajo
            else:
                profundidad = 0  # cabeza de clasificación / pooler = LR base
            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre
            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)
            grupos.setdefault(clave, []).append(p)
        self.optimizer = torch.optim.AdamW(
            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]
        )
        return self.optimizer


class ParadaTemprana(TrainerCallback):
    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""

    def __init__(self, paciencia=2):
        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_f1_macro")
        if f1 is None:
            return control
        if f1 > self.mejor:
            self.mejor, self.sin_mejora = f1, 0
        else:
            self.sin_mejora += 1
            if self.sin_mejora >= self.paciencia:
                control.should_training_stop = True
        return control


class DetenerEnEpoca(TrainerCallback):
    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""

    def __init__(self, epocas):
        self.epocas = int(epocas)

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:
            control.should_training_stop = True
        return control


In [8]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        # Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época
        # se leen de la curva. El early stopping lo hace ParadaTemprana.
        params.update({"save_strategy": "no", "load_best_model_at_end": False})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    # warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).
    if "warmup_ratio" in sig:
        params["warmup_ratio"] = float(WARMUP_FRACCION)
    else:
        params["warmup_steps"] = float(WARMUP_FRACCION)
    # nunca descartar argumentos en silencio.
    descartados = sorted(k for k in params if k not in sig)
    if descartados:
        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")
    return TrainingArguments(**params)


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

In [9]:
def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)

## 4. Focal Loss y entrenamiento del piloto

`TrainerPiloto` hereda el Trainer del notebook 04 (pesos de clase, LR por capas y, en BETO y DistilBETO, enmascarado del 15 %) y solo reemplaza la pérdida de entrenamiento cuando la opción es Focal Loss. La celda comprueba antes que, con γ = 0, Focal Loss es idéntica a la entropía cruzada de PyTorch.

In [10]:
def perdida_focal(logits, labels, gamma, weights=None, label_smoothing=0.0):
    """Focal Loss multiclase (Lin et al., 2017) con pesos de clase (α) y label smoothing opcional.

    Con gamma = 0 coincide exactamente con F.cross_entropy(weight=..., label_smoothing=...),
    incluida la normalización por la suma de los pesos de la clase real.
    """
    logp = F.log_softmax(logits.float(), dim=-1)
    k = logp.size(-1)
    objetivo = F.one_hot(labels, k).float()
    if label_smoothing > 0:
        objetivo = objetivo * (1.0 - label_smoothing) + label_smoothing / k
    modulador = (1.0 - logp.exp()).clamp(min=0.0) ** gamma
    por_clase = -objetivo * modulador * logp
    if weights is not None:
        w = weights.float().to(logp.device)
        return (por_clase * w).sum(dim=-1).sum() / w[labels].sum()
    return por_clase.sum(dim=-1).mean()


class TrainerPiloto(ClasificadorTrainer):
    """Igual que el Trainer del notebook 04; solo cambia la pérdida de ENTRENAMIENTO.

    La pérdida de validación es la misma en todas las opciones (entropía cruzada ponderada,
    sin label smoothing), así que las curvas de validación son comparables.
    """

    def __init__(self, *args, perdida="ce", gamma=0.0, **kwargs):
        self.perdida = perdida
        self.gamma = float(gamma)
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if self.perdida == "ce" or not model.training:
            return super().compute_loss(model, inputs, return_outputs=return_outputs,
                                        num_items_in_batch=num_items_in_batch, **kwargs)
        if self.mascara_tokens > 0:
            inputs = self._enmascarar(inputs)
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = perdida_focal(outputs.logits, labels, self.gamma, self.class_weights, self.label_smoothing)
        return (loss, outputs) if return_outputs else loss


# Comprobación: con gamma = 0, Focal Loss = entropía cruzada de PyTorch (con y sin pesos / smoothing).
_g = torch.Generator().manual_seed(0)
_z = torch.randn(64, 2, generator=_g)
_y = torch.randint(0, 2, (64,), generator=_g)
_w = torch.tensor([0.75, 1.5])
for _ls in (0.0, 0.1):
    for _peso in (None, _w):
        _a = perdida_focal(_z, _y, 0.0, _peso, _ls)
        _b = F.cross_entropy(_z, _y, weight=_peso, label_smoothing=_ls)
        assert torch.allclose(_a, _b, atol=1e-6), (_ls, _peso, float(_a), float(_b))
assert perdida_focal(_z, _y, 2.0, _w) < perdida_focal(_z, _y, 0.0, _w)
print("Focal Loss verificada: con γ=0 coincide con F.cross_entropy (con y sin pesos y label smoothing).")

Focal Loss verificada: con γ=0 coincide con F.cross_entropy (con y sin pesos y label smoothing).


### 4.1 Ejecutar los 36 fine-tunings

Orden: variante → opción → fold. Cada fold terminado se guarda (`.npz` + `.csv`); si se interrumpe, al volver a ejecutar se reutiliza y solo se entrena lo que falta.

In [11]:
def splits_variante(variante):
    df = TRAIN_AGRUPADO[variante].copy()
    cache, y = cache_dataset(df)
    groups = df[GROUP_COL].astype(str).to_numpy()
    # Mismos splits que la optimización final y el diagnóstico del notebook 04.
    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return cache, y, list(skf.split(np.zeros(len(y)), y, groups=groups))


def entrenar_fold(variante, opcion, numero, cache, y, tr, val):
    ruta = PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.npz"
    if ruta.exists():
        return "reutilizado"
    cfg_full = CONFIG_FINAL[variante]
    cfg = {k: v for k, v in cfg_full.items() if k != "refit_epochs"}
    op = OPCIONES[opcion]
    epocas = int(cfg["num_train_epochs"])
    ds = cache[cfg["preprocessing"]]
    seed_fold = SEED + SEED * 100 + numero  # misma semilla que la búsqueda final, igual en las 4 opciones
    carpeta = TMP_DIR / f"{opcion}_{variante}_f{numero}"
    registro = []

    def _metricas(prediction):
        logits = prediction[0][0] if isinstance(prediction[0], tuple) else prediction[0]
        registro.append(np.asarray(logits, dtype=np.float32))
        return metricas(prediction)

    model = crear_modelo(seed_fold, cfg["classifier_dropout"], cfg)
    trainer = None
    try:
        trainer = entrenador_silencioso(
            TrainerPiloto(
                model=model,
                args=argumentos_entrenamiento(carpeta, cfg, seed_fold, con_validacion=True, epochs=epocas),
                train_dataset=Subset(ds, tr.tolist()),
                eval_dataset=Subset(ds, val.tolist()),
                data_collator=data_collator,
                compute_metrics=_metricas,
                class_weights=pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None,
                label_smoothing=op["label_smoothing"],
                lr_decay=cfg.get("lr_decay", 1.0),
                mascara_tokens=cfg.get("mascara_tokens", 0.0),
                perdida=op["perdida"],
                gamma=op["gamma"],
            )
        )
        trainer.train()
        historia = trainer.state.log_history
        evals = [e for e in historia if "eval_f1_macro" in e]
        perdidas = {round(e["epoch"]): e["loss"] for e in historia if "loss" in e and "epoch" in e}
        if len(evals) != len(registro):
            raise RuntimeError(f"{opcion}/{variante}/fold{numero}: {len(evals)} evaluaciones y {len(registro)} logits.")
        tabla = pd.DataFrame({
            "opcion": opcion, "variante": variante, "fold": numero,
            "epoca": [int(round(e["epoch"])) for e in evals],
            "val_f1": [e["eval_f1_macro"] for e in evals],
            "val_nll": [e["eval_nll"] for e in evals],
            "val_ece": [e["eval_ece"] for e in evals],
            "train_loss": [perdidas.get(int(round(e["epoch"])), np.nan) for e in evals],
        })
        csv_atomico(tabla, PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.csv")
        tmp = ruta.with_name(ruta.stem + "_tmp.npz")
        np.savez_compressed(tmp, logits=np.stack(registro), labels=y[val], val_idx=np.asarray(val),
                            epocas=tabla["epoca"].to_numpy())
        os.replace(tmp, ruta)  # el .npz final solo existe si el fold terminó completo
        return "entrenado"
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
        shutil.rmtree(carpeta, ignore_errors=True)


def ejecutar_piloto():
    total = len(OPCIONES) * len(VARIANTES_PILOTO) * N_FOLDS
    hechos, t_inicio = 0, time.time()
    for variante in VARIANTES_PILOTO:
        cache, y, splits = splits_variante(variante)
        refit = int(CONFIG_FINAL[variante]["refit_epochs"])
        for opcion in OPCIONES:
            for numero, (tr, val) in enumerate(splits, start=1):
                t0 = time.time()
                estado = entrenar_fold(variante, opcion, numero, cache, y, tr, val)
                hechos += 1
                d = pd.read_csv(PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.csv")
                f1_refit = d.loc[d["epoca"] == refit, "val_f1"]
                f1_txt = f"{float(f1_refit.iloc[0]):.4f}" if len(f1_refit) else "—"
                print(f"[{hechos:>2}/{total}] {variante.upper()} | {opcion:<8} | fold {numero} | {estado:<11} | "
                      f"F1 val (época {refit}) = {f1_txt} | {time.time() - t0:5.0f} s | "
                      f"total {(time.time() - t_inicio) / 60:5.1f} min", flush=True)
    avisar_fin(True)


ejecutar_piloto()

[ 1/36] MX | CE       | fold 1 | entrenado   | F1 val (época 5) = 0.6423 |    34 s | total   0.6 min
[ 2/36] MX | CE       | fold 2 | entrenado   | F1 val (época 5) = 0.6277 |    33 s | total   1.1 min
[ 3/36] MX | CE       | fold 3 | entrenado   | F1 val (época 5) = 0.7124 |    33 s | total   1.7 min
[ 4/36] MX | FL_g1    | fold 1 | entrenado   | F1 val (época 5) = 0.6333 |    33 s | total   2.2 min
[ 5/36] MX | FL_g1    | fold 2 | entrenado   | F1 val (época 5) = 0.6261 |    33 s | total   2.8 min
[ 6/36] MX | FL_g1    | fold 3 | entrenado   | F1 val (época 5) = 0.7075 |    33 s | total   3.3 min
[ 7/36] MX | FL_g2    | fold 1 | entrenado   | F1 val (época 5) = 0.6327 |    33 s | total   3.9 min
[ 8/36] MX | FL_g2    | fold 2 | entrenado   | F1 val (época 5) = 0.6280 |    33 s | total   4.4 min
[ 9/36] MX | FL_g2    | fold 3 | entrenado   | F1 val (época 5) = 0.7049 |    35 s | total   5.0 min
[10/36] MX | FL_g2_LS | fold 1 | entrenado   | F1 val (época 5) = 0.6307 |    33 s | total 

## 5. Resultados: F1 out-of-fold y comparación pareada con CE

In [12]:
def f1_macro_np(y, p):
    return f1_score(y, p, average="macro")


def cargar_oof(variante, opcion):
    """Logits out-of-fold (los 3 folds cubren todo el train) en cada época."""
    partes = [np.load(PILOTO_DIR / f"{opcion}_{variante}_fold{n}.npz") for n in range(1, N_FOLDS + 1)]
    n = len(TRAIN_AGRUPADO[variante])
    epocas = partes[0]["epocas"]
    logits = np.full((len(epocas), n, 2), np.nan, dtype=np.float32)
    y = np.full(n, -1)
    for d in partes:
        logits[:, d["val_idx"], :] = d["logits"]
        y[d["val_idx"]] = d["labels"]
    assert (y >= 0).all() and not np.isnan(logits).any(), f"{variante}/{opcion}: OOF incompleto"
    return epocas, logits, y


rng = np.random.default_rng(SEED)
filas, filas_comp, curvas = [], [], []
for variante in VARIANTES_PILOTO:
    refit = int(CONFIG_FINAL[variante]["refit_epochs"])
    pred = {}
    for opcion in OPCIONES:
        epocas, logits, y = cargar_oof(variante, opcion)
        k = int(np.where(epocas == refit)[0][0])
        probs = softmax_np(logits[k])
        pred[opcion] = probs.argmax(axis=1)
        por_fold = pd.concat([pd.read_csv(PILOTO_DIR / f"{opcion}_{variante}_fold{n}.csv")
                              for n in range(1, N_FOLDS + 1)])
        f1_fold = por_fold[por_fold["epoca"] == refit]["val_f1"]
        media_epoca = por_fold.groupby("epoca")["val_f1"].mean()
        filas.append({
            "variante": variante, "opcion": opcion, "nombre": OPCIONES[opcion]["nombre"],
            "epoca_refit": refit,
            "f1_oof": f1_macro_np(y, pred[opcion]),
            "f1_ironia_oof": f1_score(y, pred[opcion], pos_label=1),
            "f1_folds_media": f1_fold.mean(), "f1_folds_de": f1_fold.std(),
            "nll_oof": nll_binaria(probs, y), "ece_oof": ece_binaria(probs, y),
            "mejor_epoca_media": int(media_epoca.idxmax()), "f1_mejor_epoca_media": media_epoca.max(),
        })
        for e, logit_e in zip(epocas, logits):
            curvas.append({"variante": variante, "opcion": opcion, "epoca": int(e),
                           "f1_oof": f1_macro_np(y, softmax_np(logit_e).argmax(axis=1))})
    # Comparación pareada con la opción base (mismos tweets, mismos folds y semillas).
    n = len(y)
    muestras = rng.integers(0, n, size=(N_BOOTSTRAP, n))
    for opcion in OPCIONES:
        if opcion == BASE:
            continue
        dif = f1_macro_np(y, pred[opcion]) - f1_macro_np(y, pred[BASE])
        boot = np.array([f1_macro_np(y[i], pred[opcion][i]) - f1_macro_np(y[i], pred[BASE][i]) for i in muestras])
        folds_mejor = sum(
            pd.read_csv(PILOTO_DIR / f"{opcion}_{variante}_fold{f}.csv").set_index("epoca").loc[refit, "val_f1"]
            > pd.read_csv(PILOTO_DIR / f"{BASE}_{variante}_fold{f}.csv").set_index("epoca").loc[refit, "val_f1"]
            for f in range(1, N_FOLDS + 1)
        )
        filas_comp.append({
            "variante": variante, "opcion": opcion, "dif_vs_CE": dif,
            "ic95_inf": np.percentile(boot, 2.5), "ic95_sup": np.percentile(boot, 97.5),
            "prop_boot_mejor": float((boot > 0).mean()), "folds_mejor": f"{folds_mejor}/{N_FOLDS}",
            "mejora_clara": bool(np.percentile(boot, 2.5) > 0),
        })

RESUMEN = pd.DataFrame(filas)
COMPARACION = pd.DataFrame(filas_comp)
CURVAS = pd.DataFrame(curvas)
base = NOMBRE_EXPERIMENTO.lower()
csv_atomico(RESUMEN, PILOTO_DIR / f"{base}_piloto_focal_resumen.csv")
csv_atomico(COMPARACION, PILOTO_DIR / f"{base}_piloto_focal_comparacion.csv")
csv_atomico(CURVAS, PILOTO_DIR / f"{base}_piloto_focal_curvas.csv")

print("F1-Macro OOF en la época del refit (train oficial, 3 folds agrupados):")
display(RESUMEN.pivot(index="opcion", columns="variante", values="f1_oof")
        .reindex(index=list(OPCIONES), columns=VARIANTES_PILOTO).round(4))
print("\nDiferencia con CE e intervalo bootstrap pareado del 95 %:")
display(COMPARACION.round(4))
print("\nDetalle (F1 por fold, calibración y mejor época):")
display(RESUMEN.drop(columns=["nombre"]).round(4))

F1-Macro OOF en la época del refit (train oficial, 3 folds agrupados):


variante,mx,es,cu
opcion,,,
CE,0.6609,0.7333,0.7206
FL_g1,0.6557,0.7315,0.7170
FL_g2,0.6553,0.7334,0.7143
FL_g2_LS,0.6509,0.7320,0.7172



Diferencia con CE e intervalo bootstrap pareado del 95 %:


,variante,opcion,dif_vs_CE,ic95_inf,ic95_sup,prop_boot_mejor,folds_mejor,mejora_clara
0,mx,FL_g1,-0.0052,-0.0131,0.0023,0.0885,0/3,False
1,mx,FL_g2,-0.0056,-0.0136,0.0032,0.1140,1/3,False
2,mx,FL_g2_LS,-0.0099,-0.0185,-0.0017,0.0120,0/3,False
3,es,FL_g1,-0.0018,-0.0088,0.0052,0.3280,0/3,False
4,es,FL_g2,0.0001,-0.0082,0.0083,0.5075,2/3,False
5,es,FL_g2_LS,-0.0013,-0.0086,0.0063,0.3690,1/3,False
6,cu,FL_g1,-0.0035,-0.0120,0.0044,0.1915,1/3,False
7,cu,FL_g2,-0.0063,-0.0167,0.0034,0.1040,1/3,False
8,cu,FL_g2_LS,-0.0033,-0.0125,0.0055,0.2345,2/3,False



Detalle (F1 por fold, calibración y mejor época):


,variante,opcion,epoca_refit,f1_oof,f1_ironia_oof,f1_folds_media,f1_folds_de,nll_oof,ece_oof,mejor_epoca_media,f1_mejor_epoca_media
0,mx,CE,5,0.6609,0.5851,0.6608,0.0453,0.6212,0.0648,5,0.6608
1,mx,FL_g1,5,0.6557,0.5684,0.6556,0.0450,0.5899,0.0251,6,0.6582
2,mx,FL_g2,5,0.6553,0.5656,0.6552,0.0431,0.5948,0.0411,4,0.6607
3,mx,FL_g2_LS,5,0.6509,0.5724,0.6510,0.0467,0.6192,0.0659,6,0.6517
4,es,CE,6,0.7333,0.6548,0.7333,0.0088,0.5003,0.0379,6,0.7333
5,es,FL_g1,6,0.7315,0.6472,0.7315,0.0093,0.4822,0.0302,4,0.7317
6,es,FL_g2,6,0.7334,0.6483,0.7334,0.0077,0.4958,0.0612,6,0.7334
7,es,FL_g2_LS,6,0.7320,0.6520,0.7320,0.0123,0.5460,0.1209,6,0.7320
8,cu,CE,6,0.7206,0.6311,0.7203,0.0162,0.5228,0.0294,6,0.7203
9,cu,FL_g1,6,0.7170,0.6174,0.7168,0.0191,0.5116,0.0290,6,0.7168


In [13]:
fig, axes = plt.subplots(1, len(VARIANTES_PILOTO), figsize=(4.8 * len(VARIANTES_PILOTO), 3.8), sharey=True,
                         squeeze=False)
estilos = {"CE": ("k", "-"), "FL_g1": ("C0", "--"), "FL_g2": ("C3", "--"), "FL_g2_LS": ("C2", ":")}
for ax, variante in zip(axes[0], VARIANTES_PILOTO):
    for opcion in OPCIONES:
        d = CURVAS[(CURVAS["variante"] == variante) & (CURVAS["opcion"] == opcion)]
        color, linea = estilos.get(opcion, ("C7", "-"))
        ax.plot(d["epoca"], d["f1_oof"], color=color, linestyle=linea, marker="o", markersize=3,
                label=OPCIONES[opcion]["nombre"])
    ax.axvline(CONFIG_FINAL[variante]["refit_epochs"], color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"{variante.upper()} · línea gris = época del refit", fontsize=10)
    ax.set_xlabel("Época")
    ax.grid(alpha=0.25)
axes[0, 0].set_ylabel("F1-Macro OOF (train)")
axes[0, 0].legend(fontsize=7)
fig.suptitle(f"{NOMBRE_EXPERIMENTO} — piloto Focal Loss vs. Cross Entropy (3 folds agrupados, solo train)",
             fontsize=11)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(PILOTO_DIR / f"{NOMBRE_EXPERIMENTO.lower()}_piloto_focal_curvas.{ext}", dpi=200, bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1440x380 with 3 Axes>

### 5.1 Decisión (regla fijada antes de ejecutar)

In [14]:
ganadoras = (COMPARACION.groupby("opcion")["mejora_clara"].sum()
             .reindex([o for o in OPCIONES if o != BASE]).fillna(0).astype(int))
print("Variantes con mejora clara (IC 95 % > 0) frente a CE:")
for opcion, n in ganadoras.items():
    print(f"  {opcion:<9} {n}/{len(VARIANTES_PILOTO)}  -> {'SE ADOPTA' if n >= 2 else 'no se adopta'}")
adoptadas = ganadoras[ganadoras >= 2]
if adoptadas.empty:
    print(f"\nConclusión: ninguna opción de Focal Loss mejora de forma clara a la entropía cruzada ponderada en "
          f"{NOMBRE_EXPERIMENTO}. Se mantiene la función de error actual.")
else:
    mejor = COMPARACION[COMPARACION["opcion"].isin(adoptadas.index)].groupby("opcion")["dif_vs_CE"].mean().idxmax()
    print(f"\nConclusión: {mejor} mejora de forma clara en al menos 2 de 3 variantes. Siguiente paso: fijar esa "
          "función de error y repetir el entrenamiento final y el test del notebook 04.")

Variantes con mejora clara (IC 95 % > 0) frente a CE:
  FL_g1     0/3  -> no se adopta
  FL_g2     0/3  -> no se adopta
  FL_g2_LS  0/3  -> no se adopta

Conclusión: ninguna opción de Focal Loss mejora de forma clara a la entropía cruzada ponderada en BETO. Se mantiene la función de error actual.


## 5. Cómo leer los resultados

- `f1_oof`: F1-Macro de las predicciones *out-of-fold* (todo el train, 3 folds) en la época del refit. Es la cifra principal de comparación.
- `dif_vs_CE`, `ic95_inf`, `ic95_sup`: diferencia con la función de error actual y su intervalo bootstrap pareado del 95 %. Si el intervalo incluye el 0, la diferencia no es concluyente.
- `folds_mejor`: en cuántos de los 3 folds la opción supera a `CE` (misma partición y misma semilla).
- `nll_oof`, `ece_oof`: calibración de las probabilidades antes del *temperature scaling*. Focal Loss suele dar probabilidades menos extremas; en el modelo final el *temperature scaling* las recalibra.
- `mejor_epoca_media`: época con mayor F1 medio de validación; si Focal Loss la desplaza mucho respecto del refit, se indica en el informe.

Los archivos quedan en `data/pilotos_focal_loss/<modelo>/`: los `.npz` y `.csv` por opción, variante y fold, y `<modelo>_piloto_focal_resumen.csv`, `_comparacion.csv`, `_curvas.csv` y `_curvas.png/.pdf`. El notebook `Piloto_FocalLoss_Resumen.ipynb` reúne los tres modelos.